<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/4_model_training/4_0_model_training_plan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 4_0_model_training_plan


## Introducción y Resumen

El objetivo de esta notebook es generar las ventanas de entrada (X) y targets (y) para los horizontes de 30, 60 y 90 minutos, aplica el escalado de features sobre cada conjunto (train, valid, test) y finalmente guarda las ventanas escaladas en disco, dejándolas listas para el entrenamiento de modelos.

0. Configuración del Entorno

    Se conecta Google Drive y se clona el repositorio de trabajo. Se instalan e importan librerías necesarias como pandas, numpy, matplotlib y seaborn. Se cargan los datasets procesados previamente y se muestra un resumen de la información de los datasets.

1. Carga de datos

    Se importan los datasets procesados `mnq_train`, `mnq_valid` y `mnq_test`. Se revisa su estructura (filas, columnas, tipos de datos) y también de importa el listado de features seleccionados para cada ventana de tiempo.





## 0. Configuración del Entorno


### 0.1. Clonado de repositorio / Acceso a Drive

In [1]:
#Clonamos el repo
#LINK DE REPOSITORIO: https://github.com/GUNAPILLCO/neural_profit
#!git clone https://github.com/GUNAPILLCO/neural_profit.git

In [2]:
from google.colab import drive
drive.mount('/content/drive')
drive_path = "/content/drive/MyDrive/neural_profit"

Mounted at /content/drive


### 0.2. Instalación de librerías


In [3]:
#!{sys.executable} -m pip install -q ta
#print("Librería instalada: technical-analysis")

### 0.3. Importación de librerías


In [4]:
import sys
import re
#Instalación de librería pandas_market_calendars
#!{sys.executable} -m pip install -q pandas_market_calendars
#print("Librería instalada: pandas_market_calendars")


from functools import reduce
# Utilidades generales
from datetime import datetime, timedelta
import os
import glob
import requests
import warnings
warnings.filterwarnings('ignore')

# Manejo y procesamiento de datos
#import ta
import pandas as pd
import numpy as np
from tabulate import tabulate
import matplotlib.pyplot as plt
# Calendario de mercados
#import pandas_market_calendars as mcal

#from ta.momentum import StochasticOscillator, ROCIndicator
#from ta.volatility import BollingerBands, AverageTrueRange

from scipy.stats import spearmanr
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import joblib

## 1. Carga de datos

### 1.1. Carga de datasets `mnq_train`, `mnq_valid` y `mnq_test`






In [5]:
def load_data(data: str):

    data_path = f'{drive_path}/3_dataset_preparation/mnq_{data}.parquet'
    # Leer el archivo Parquet y cargarlo en un DataFrame
    df = pd.read_parquet(data_path)

    # Asegurar que el índice esté en formato datetime
    df.index = pd.to_datetime(df.index)

    # Crear una nueva columna 'date' con la fecha extraída del índice
    df['date'] = df.index.date

    # Reordenar columnas: 'date', 'time_str', y luego el resto
    cols = ['date'] + [col for col in df.columns if col not in ['date']]

    df = df[cols]

    return df

In [6]:
#mnq_train = load_data("train")
#mnq_valid = load_data("valid")
#mnq_test = load_data("test")

### 1.2. Información de datasets


In [7]:
def info_dataset(df, name: str):
  print(f"Información del dataset {name}:\n")

  # Contar valores únicos en la columna 'date'
  num_dias = df['date'].nunique()
  print(f"\tCantidad de días: {num_dias}")

  # Filtrar valores válidos
  validos_por_dia = df.dropna(subset=['close']).groupby('date').size()

  # Calcular el promedio
  promedio_por_fecha = validos_por_dia.mean()
  print(f"\tRegistros por día: {int(promedio_por_fecha)}")

  primer_hora = df.index[0].strftime('%H:%M')
  ultima_hora = df.index[-1].strftime('%H:%M')
  zona_horaria = df.index[0].tzinfo


  print(f"\tHora diaria de inicio {primer_hora}")
  print(f"\tHora diaria de final {ultima_hora}")
  print(f"\tZona horaria: {zona_horaria}\n")

  return num_dias, promedio_por_fecha

In [8]:
#info_dataset(mnq_train, 'mnq_train')
#info_dataset(mnq_valid, 'mnq_valid')
#info_dataset(mnq_test, 'mnq_test')

### 1.3. Carga de listado de features por ventana de tiempo

In [9]:
import json

# Ruta al archivo guardado
path = f'{drive_path}/2_feature_engineering/features_list.json'

#with open(path, "r") as f:
#    features_dict = json.load(f)

# Extraer las listas
#features_to_30 = features_dict["features_to_30"]
#features_to_60 = features_dict["features_to_60"]
#features_to_90 = features_dict["features_to_90"]


In [10]:
#print(f'Listado de features para 30min: {features_to_30}')
#print(f'Listado de features para 60min: {features_to_60}')
#print(f'Listado de features para 90min: {features_to_90}')

## 2. Carga de ventanas `X_train_*_scaled`, `X_valid_*_scaled`, `X_test_*_scaled`

### 2.0. Funciones

#### Función para cargar ventanas

In [11]:
def load_windows_and_scaler(target: str, scaled=True):
    """
    Carga datasets (X, y) para train, valid y test junto con el scaler global.

    Parámetros
    ----------
    drive_path : str
        Ruta base donde se encuentran los archivos.
    scaled : bool, default=True
        Si True busca en la carpeta 'ventanas_x_y_scaled',
        si False en 'ventanas_x_y'.

    Retorna
    -------
    X_train, y_train, X_valid, y_valid, X_test, y_test, scaler
    """

    #Ruta de ventandas escaladas
    path_train  = f'{drive_path}/3_dataset_preparation/xy_windows_scaled/xy_train_{target}_scaled.npz'
    path_valid  = f'{drive_path}/3_dataset_preparation/xy_windows_scaled/xy_valid_{target}_scaled.npz'
    path_test   = f'{drive_path}/3_dataset_preparation/xy_windows_scaled/xy_test_{target}_scaled.npz'

    #Ruta de escalador
    path_scaler = f"{drive_path}/3_dataset_preparation/global_scaler_{target}.pkl"

    # Cargar npz
    data_train = np.load(path_train)
    data_valid = np.load(path_valid)
    data_test  = np.load(path_test)

    # Extraer X, y
    X_train, y_train = data_train["X"], data_train["y"]
    X_valid, y_valid = data_valid["X"], data_valid["y"]
    X_test,  y_test  = data_test["X"],  data_test["y"]

    # Cargar scaler
    scaler = joblib.load(path_scaler)

    return X_train, y_train, X_valid, y_valid, X_test, y_test, scaler


#### Función para revisar información de ventanas

In [12]:
def xy_info(target: str, X_train, y_train, X_valid, y_valid, X_test, y_test):
    print(f'Información para horizonte de {target} minutos:')

    for name, X, y in [
        ("entrenamiento", X_train, y_train),
        ("validación", X_valid, y_valid),
        ("testeo", X_test, y_test),
    ]:
        print(f'\nSet de {name}:')
        print(f'\t{X.shape[0]} ventanas (n_samples).')

        if X.ndim == 2:

            print(f'\t{X.shape[1]} features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_{target})')
        elif X.ndim == 3:

            print(f'\t{X.shape[1]} pasos en lookback × {X.shape[2]} features por paso. Dimensión 3D: (n_samples, window_size, len(features_{target})')

        print(f'\t{y.shape[0]} targets.')
        print(f'\tDistribución y: mean={y.mean():.6f}, std={y.std():.6f}, min={y.min():.6f}, max={y.max():.6f}')

### 2.1 Carga de ventanas 30 minutos

In [13]:
X_train_30_scaled, y_train_30, X_valid_30_scaled, y_valid_30, X_test_30_scaled, y_test_30, scaler_30 = load_windows_and_scaler(target = '30')

In [14]:
xy_info( '30', X_train_30_scaled, y_train_30, X_valid_30_scaled, y_valid_30, X_test_30_scaled, y_test_30)

Información para horizonte de 30 minutos:

Set de entrenamiento:
	193487 ventanas (n_samples).
	900 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_30)
	193487 targets.
	Distribución y: mean=0.000056, std=0.002760, min=-0.028931, max=0.032372

Set de validación:
	41567 ventanas (n_samples).
	900 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_30)
	41567 targets.
	Distribución y: mean=0.000103, std=0.003273, min=-0.023167, max=0.068056

Set de testeo:
	41567 ventanas (n_samples).
	900 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_30)
	41567 targets.
	Distribución y: mean=-0.000023, std=0.002407, min=-0.015982, max=0.016097


### 2.2 Carga de ventanas 60 minutos

In [15]:
X_train_60_scaled, y_train_60, X_valid_60_scaled, y_valid_60, X_test_60_scaled, y_test_60, scaler_60 = load_windows_and_scaler(target = '60')

In [16]:
xy_info( '60', X_train_60_scaled, y_train_60, X_valid_60_scaled, y_valid_60, X_test_60_scaled, y_test_60)

Información para horizonte de 60 minutos:

Set de entrenamiento:
	193487 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_60)
	193487 targets.
	Distribución y: mean=0.000114, std=0.003907, min=-0.040179, max=0.036224

Set de validación:
	41567 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_60)
	41567 targets.
	Distribución y: mean=0.000163, std=0.004643, min=-0.038084, max=0.079896

Set de testeo:
	41567 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_60)
	41567 targets.
	Distribución y: mean=-0.000068, std=0.003521, min=-0.018494, max=0.020365


### 2.3 Carga de ventanas 90 minutos

In [17]:
X_train_90_scaled, y_train_90, X_valid_90_scaled, y_valid_90, X_test_90_scaled, y_test_90, scaler_90 = load_windows_and_scaler(target = '90')

In [18]:
xy_info('90', X_train_90_scaled, y_train_90, X_valid_90_scaled, y_valid_90, X_test_90_scaled, y_test_90)

Información para horizonte de 90 minutos:

Set de entrenamiento:
	193487 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_90)
	193487 targets.
	Distribución y: mean=0.000159, std=0.004746, min=-0.037742, max=0.039284

Set de validación:
	41567 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_90)
	41567 targets.
	Distribución y: mean=0.000224, std=0.005887, min=-0.040748, max=0.083184

Set de testeo:
	41567 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_90)
	41567 targets.
	Distribución y: mean=-0.000132, std=0.004558, min=-0.023392, max=0.026213


## 3. Carga de métricas


In [19]:
def load_metrics(data: str):
    data_path = f'{drive_path}/4_model_training/{data}.parquet'
    # Leer el archivo Parquet y cargarlo en un DataFrame
    df = pd.read_parquet(data_path)
    return df

def metrics_verify(data: str) -> bool:
    data_path = f"{drive_path}/4_model_training/{data}.parquet"
    return os.path.exists(data_path)

def load_or_create_metrics (data:str):
  if metrics_verify(data):
      print(f"Las métricas existen y son almacenadas en {data[4:len(data)]}")
      model_metrics = load_metrics(data)
      #print(random_forest_metrics)
      metrics = True
  else:
      print(f"Las métricas no existen. Se crea el dataset {data[4:len(data)]} para almacenar las métricas")
      #Creamos la tabla para almacenar las métricas
      model_metrics = pd.DataFrame(columns=["RMSE", "MAE", "R2", "SMAPE", "DirAcc"])
      metrics = False

  return model_metrics, metrics


In [20]:
rf_metrics, metrics   =  load_or_create_metrics("4_1_random_forest_metrics")
xgb_metrics, metrics  =  load_or_create_metrics("4_2_xgboost_metrics")
lgbm_metrics, metrics =  load_or_create_metrics("4_3_lgbm_metrics")
cat_metrics, metrics  =  load_or_create_metrics("4_4_catboost_metrics")
mlp_metrics, metrics  =  load_or_create_metrics("4_5_mlp_metrics")
tcn_metrics, metrics =  load_or_create_metrics("4_6_tcn_metrics")
transformers_metrics, metrics  =  load_or_create_metrics("4_7_transformers_metrics")
#tabnet_metrics        =  load_or_create_metrics("4_8_tabnet_metrics")
lstm_metrics          =  load_or_create_metrics("4_8_lstm_metrics")
gru_metrics           =  load_or_create_metrics("4_9_gru_metrics")


Las métricas existen y son almacenadas en random_forest_metrics
Las métricas existen y son almacenadas en xgboost_metrics
Las métricas existen y son almacenadas en lgbm_metrics
Las métricas existen y son almacenadas en catboost_metrics
Las métricas existen y son almacenadas en mlp_metrics
Las métricas existen y son almacenadas en tcn_metrics
Las métricas existen y son almacenadas en transformers_metrics
Las métricas existen y son almacenadas en lstm_metrics
Las métricas existen y son almacenadas en gru_metrics


In [29]:
transformers_metrics  #pandas.core.frame.DataFrame

,RMSE,MAE,R2,SMAPE,DirAcc
transformer_30_100%,0.002053,0.001415,0.452970,111.805275,0.720148
transformer_60_100%,0.002073,0.001432,0.712301,86.124934,0.810990
transformer_90_100%,0.002221,0.001489,0.783293,77.708278,0.841964


In [32]:
lstm_metrics = lstm_metrics[0] #tuple

In [34]:
gru_metrics = gru_metrics[0] #tuple

In [35]:
# Combinar verticalmente
model_metrics = pd.concat([rf_metrics, xgb_metrics, lgbm_metrics, cat_metrics, mlp_metrics, tcn_metrics, transformers_metrics, lstm_metrics, gru_metrics], axis=0)


In [36]:
model_metrics

,RMSE,MAE,R2,SMAPE,DirAcc
RF_30_subsampleado_30%,0.002768,0.001628,0.237195,120.874522,0.698236
RF_30_subsampleado_50%,0.002782,0.001640,0.233573,121.386628,0.696868
RF_60_subsampleado_30%,0.003737,0.002029,0.326394,105.691331,0.760706
RF_60_subsampleado_50%,0.003946,0.002072,0.300929,105.248839,0.763990
RF_90_subsampleado_30%,0.004722,0.002363,0.352733,96.445722,0.794066
...,...,...,...,...,...
GRU_60_100%,0.004141,0.002422,0.204701,120.918488,0.683379
GRU_90_subsampleado_30%,0.005309,0.002834,0.251955,110.117842,0.739374
GRU_90_subsampleado_50%,0.004951,0.002877,0.243213,110.747967,0.727806
GRU_90_subsampleado_50%_dropout_wd,0.005212,0.002861,0.223817,114.636358,0.724919
